# Feature feasibility audit

This section checks additional national Eurostat features before they are merged into the panel.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from eurostat_api import build_country_year_table, download_json, eurostat_url, is_national_geo

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
YEARS = range(2008, 2026)
outcome = pd.read_csv(PROCESSED_DIR / 'panel_skeleton.csv')
outcome_pairs = set(zip(outcome['geo'], outcome['year']))
outcome.shape

(608, 9)

In [2]:
candidates = [
    {
        'dataset_code': 'hlth_rs_prs2',
        'short_title': 'Physicians per 100 000 inhabitants',
        'variable_name': 'physicians_per_100k',
        'params': {'unit': 'P_HTHAB', 'wstatus': 'PRACT', 'med_spec': 'PHYS'},
        'filter': 'unit=P_HTHAB, wstatus=PRACT, med_spec=PHYS',
    },
    {
        'dataset_code': 'hlth_rs_bds1',
        'short_title': 'Hospital beds per 100 000 inhabitants',
        'variable_name': 'hospital_beds_per_100k',
        'params': {'unit': 'P_HTHAB', 'facility': 'HBEDT', 'hlthcare': 'TOTAL'},
        'filter': 'unit=P_HTHAB, facility=HBEDT, hlthcare=TOTAL',
    },
    {
        'dataset_code': 'hlth_sha11_hf',
        'short_title': 'Out-of-pocket health expenditure share',
        'variable_name': 'oop_health_expenditure_share_pc',
        'params': {'unit': 'PC_CHE', 'icha11_hf': 'HF3'},
        'filter': 'unit=PC_CHE, icha11_hf=HF3',
    },
    {
        'dataset_code': 'ilc_di12',
        'short_title': 'Gini coefficient of income inequality',
        'variable_name': 'gini_income',
        'params': {'age': 'TOTAL', 'statinfo': 'GINI_HND'},
        'filter': 'age=TOTAL, statinfo=GINI_HND',
    },
    {
        'dataset_code': 'une_ltu_a',
        'short_title': 'Long-term unemployment rate',
        'variable_name': 'long_term_unemployment_rate_pc',
        'params': {'indic_em': 'LTU', 'age': 'Y15-74', 'sex': 'T', 'unit': 'PC_ACT'},
        'filter': 'indic_em=LTU, age=Y15-74, sex=T, unit=PC_ACT',
    },
]
candidates

[{'dataset_code': 'hlth_rs_prs2',
  'short_title': 'Physicians per 100 000 inhabitants',
  'variable_name': 'physicians_per_100k',
  'params': {'unit': 'P_HTHAB', 'wstatus': 'PRACT', 'med_spec': 'PHYS'},
  'filter': 'unit=P_HTHAB, wstatus=PRACT, med_spec=PHYS'},
 {'dataset_code': 'hlth_rs_bds1',
  'short_title': 'Hospital beds per 100 000 inhabitants',
  'variable_name': 'hospital_beds_per_100k',
  'params': {'unit': 'P_HTHAB', 'facility': 'HBEDT', 'hlthcare': 'TOTAL'},
  'filter': 'unit=P_HTHAB, facility=HBEDT, hlthcare=TOTAL'},
 {'dataset_code': 'hlth_sha11_hf',
  'short_title': 'Out-of-pocket health expenditure share',
  'variable_name': 'oop_health_expenditure_share_pc',
  'params': {'unit': 'PC_CHE', 'icha11_hf': 'HF3'},
  'filter': 'unit=PC_CHE, icha11_hf=HF3'},
 {'dataset_code': 'ilc_di12',
  'short_title': 'Gini coefficient of income inequality',
  'variable_name': 'gini_income',
  'params': {'age': 'TOTAL', 'statinfo': 'GINI_HND'},
  'filter': 'age=TOTAL, statinfo=GINI_HND'},


In [3]:
def category_label(data, dim_id, code):
    labels = data.get('dimension', {}).get(dim_id, {}).get('category', {}).get('label', {})
    return labels.get(code, code)

def endpoint_available(dataset_code):
    url = f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/structure/dataflow/ESTAT/{dataset_code}/1.0?compress=false'
    response = requests.get(url, timeout=45)
    return 'yes' if response.ok else 'no'

rows = []
metadata_rows = []
for candidate in candidates:
    params = dict(candidate['params'])
    params['time'] = [str(year) for year in YEARS]
    raw_path = RAW_DIR / f"feature_{candidate['dataset_code']}_{candidate['variable_name']}.json"
    data = download_json(candidate['dataset_code'], params, raw_path)
    tidy = build_country_year_table(data, candidate['variable_name'], YEARS)
    tidy = tidy[['geo', 'year', candidate['variable_name']]]
    tidy.to_csv(PROCESSED_DIR / f"feature_{candidate['variable_name']}.csv", index=False)

    national_codes = sorted(code for code in tidy['geo'].dropna().unique() if is_national_geo(str(code)))
    years_available = sorted(int(year) for year in tidy['year'].dropna().unique())
    feature_pairs = set(zip(tidy['geo'], tidy['year']))
    overlap_pairs = feature_pairs.intersection(outcome_pairs)
    unit_code = str(candidate['params'].get('unit', ''))
    unit_label = category_label(data, 'unit', unit_code) if unit_code else category_label(data, 'statinfo', str(candidate['params'].get('statinfo', '')))
    official_title = data.get('label', candidate['short_title'])
    years_text = f"{min(years_available)}-{max(years_available)}" if years_available else 'none'
    geography = 'national country codes' if national_codes else 'not verified'
    overlap = 'yes' if overlap_pairs else 'no'
    missing_cells = len(outcome_pairs) - len(overlap_pairs)
    rows.append({
        'dataset_code': candidate['dataset_code'],
        'short_title': candidate['short_title'],
        'official_title': official_title,
        'variable_name': candidate['variable_name'],
        'filter': candidate['filter'],
        'unit': unit_label,
        'geography_level': geography,
        'years_available': years_text,
        'countries_available': len(national_codes),
        'overlaps_with_outcome': overlap,
        'non_missing_country_years': len(tidy),
        'overlap_country_years': len(overlap_pairs),
        'missingness_notes': f'{len(overlap_pairs)} observed outcome cells match this exact filter; {missing_cells} outcome cells would be missing after a left merge.',
        'dataflow_endpoint_available': endpoint_available(candidate['dataset_code']),
        'merge_approved': 'yes' if overlap == 'yes' and len(overlap_pairs) >= 250 else 'no',
    })
    metadata_rows.append({
        'dataset_code': candidate['dataset_code'],
        'variable_name': candidate['variable_name'],
        'api_url': eurostat_url(candidate['dataset_code'], params),
        'raw_file': str(raw_path.relative_to(PROJECT_ROOT)),
    })

feasibility = pd.DataFrame(rows)
feasibility.to_csv(OUTPUTS_DIR / 'control_feasibility_extended.csv', index=False)
(OUTPUTS_DIR / 'feature_api_requests.json').write_text(json.dumps(metadata_rows, indent=2), encoding='utf-8')
feasibility

,dataset_code,short_title,official_title,variable_name,filter,unit,geography_level,years_available,countries_available,overlaps_with_outcome,non_missing_country_years,overlap_country_years,missingness_notes,dataflow_endpoint_available,merge_approved
0,hlth_rs_prs2,Physicians per 100 000 inhabitants,Health personnel,physicians_per_100k,"unit=P_HTHAB, wstatus=PRACT, med_spec=PHYS",Per hundred thousand inhabitants,national country codes,2008-2024,30,yes,459,431,431 observed outcome cells match this exact fi...,yes,yes
1,hlth_rs_bds1,Hospital beds per 100 000 inhabitants,Hospital beds by function and type of care,hospital_beds_per_100k,"unit=P_HTHAB, facility=HBEDT, hlthcare=TOTAL",Per hundred thousand inhabitants,national country codes,2008-2024,37,yes,570,522,522 observed outcome cells match this exact fi...,yes,yes
2,hlth_sha11_hf,Out-of-pocket health expenditure share,Health care expenditure by financing scheme,oop_health_expenditure_share_pc,"unit=PC_CHE, icha11_hf=HF3",Percentual share of total current health expen...,national country codes,2008-2024,37,yes,475,444,444 observed outcome cells match this exact fi...,yes,yes
3,ilc_di12,Gini coefficient of income inequality,Gini coefficient of equivalised disposable inc...,gini_income,"age=TOTAL, statinfo=GINI_HND",Gini coefficient (scale from 0 to 100),national country codes,2014-2025,37,yes,410,409,409 observed outcome cells match this exact fi...,yes,yes
4,une_ltu_a,Long-term unemployment rate,Long-term unemployment by sex - annual data,long_term_unemployment_rate_pc,"indic_em=LTU, age=Y15-74, sex=T, unit=PC_ACT",Percentage of population in the labour force,national country codes,2008-2025,35,yes,572,554,554 observed outcome cells match this exact fi...,yes,yes


In [4]:
approved = feasibility.loc[feasibility['merge_approved'].eq('yes'), 'variable_name'].tolist()
summary = (
    '# Extended control feasibility summary\n\n'
    f'The audit checked {len(feasibility)} additional national Eurostat features. '
    f'{len(approved)} features passed the overlap rule and are approved for the Phase 4 panel: '
    + (', '.join(approved) if approved else 'none')
    + '.\n\nThe table records exact filters, units, national coverage, year coverage, and overlap with the observed outcome panel. '
    'Features that do not meet the overlap rule are left out of the merged feature panel.\n\n'
    + feasibility.to_markdown(index=False)
    + '\n'
)
(OUTPUTS_DIR / 'control_feasibility_extended.md').write_text(summary, encoding='utf-8')
approved

['physicians_per_100k',
 'hospital_beds_per_100k',
 'oop_health_expenditure_share_pc',
 'gini_income',
 'long_term_unemployment_rate_pc']